In [0]:
# Configuration

CATALOG = "workspace"
DATABASE = "qualite_eau"

df_silver = spark.table(f"{CATALOG}.{DATABASE}.silver_qualite_eau")
total = df_silver.count()
print(f"Lignes à valider : {total:,}")

In [0]:
# Validations

from pyspark.sql import functions as F
from datetime import datetime

resultats = []

def check(nom, condition, mostly=1.0):
    ok = df_silver.filter(condition).count()
    taux = ok / total
    succes = taux >= mostly
    resultats.append({
        "regle": nom,
        "nb_ok": ok,
        "nb_total": total,
        "taux_pct": round(taux * 100, 2),
        "seuil_pct": mostly * 100,
        "succes": succes
    })
    status = "✅" if succes else "❌"
    print(f"{status} {nom} : {ok:,}/{total:,} ({taux*100:.2f}%)")
    return succes

print("📋 Validation Silver :\n")
check("code_commune non null", F.col("code_commune").isNotNull())
check("date_prelevement non null", F.col("date_prelevement").isNotNull())
check("libelle_parametre non null", F.col("libelle_parametre").isNotNull())
check("code_commune format 5 chiffres", F.col("code_commune").rlike(r"^\d{5}$"), mostly=0.99)
check("categorie_parametre valide", F.col("categorie_parametre").isin(["microbiologie", "chimie", "radioactivite", "autre"]))
check("annee entre 2020 et 2027", F.col("annee").between(2020, 2027), mostly=0.99)
check("est_conforme renseigné à 80%", F.col("est_conforme").isNotNull(), mostly=0.80)
check("resultat_numerique >= 0", F.col("resultat_numerique").isNull() | (F.col("resultat_numerique") >= 0), mostly=0.99)
check("mois entre 1 et 12", F.col("mois").between(1, 12), mostly=0.99)

In [0]:
# Résumé et sauvegarde

nb_ok = sum(1 for r in resultats if r["succes"])
nb_ko = len(resultats) - nb_ok

print(f"\n{'='*50}")
print(f"Résultat : {'✅ SUCCÈS' if nb_ko == 0 else '❌ ÉCHEC'}")
print(f"{'='*50}")
print(f"Règles OK : {nb_ok} / {len(resultats)}")
print(f"Règles KO : {nb_ko} / {len(resultats)}")

# Sauvegarde rapport
df_rapport = spark.createDataFrame(resultats)
(df_rapport.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{DATABASE}.gold_rapport_qualite"))

print("\n✅ Rapport sauvegardé dans gold_rapport_qualite")
display(df_rapport)